# Optimizing Python Code for Data Science

Part 2 of the [Efficient Python series](./README.md). Read the [complete article](https://todatabeyond.com/blog/write-efficient-python-code-for-data-scientists-optimizing-your-code).

Run cells in order in Python 3.11+ / IPython, on CPU. No API keys or GPU. Setup installs pinned packages from PyPI. Both datasets download over certificate-verified HTTPS from the original author’s repository at an immutable commit and are SHA-256 checked. The author CSV contains 800 Pokémon (not the 720 mentioned in the original prose) and 1,232 baseball rows.

Retains all 48 source gist examples and supplies the iterrows run-difference example described in the article. Repairs missing inputs, two indentation defects and the loop-variable collision that would overwrite the Pokémon DataFrame. Timing functions return results instead of printing inside repetitions; demonstrations print once outside. Timings use bounded real IPython timeit, with no speed/ranking assertions. Full datasets are used for computations; table and namedtuple demonstrations display five rows. Both slower and faster approaches remain.

Measured outputs are captured in the blog, not committed to the notebook. Timing rankings are machine/input dependent; set operations do not preserve list order or duplicate semantics.


In [ ]:
%pip install --quiet numpy==2.3.3 pandas==2.3.3 certifi==2025.10.5
from importlib.metadata import version
print("NumPy:", version("numpy"), "pandas:", version("pandas"))


In [ ]:
import hashlib
import io
import ssl
from urllib.request import urlopen
import certifi
import numpy as np
import pandas as pd

DATASET_BASE = "https://raw.githubusercontent.com/youssefHosni/Advanced-Python-Programming-Tutorials-/2507c7f72e3cb08ef349047a4ca052bcc7307337/Datasets/"
DATASET_HASHES = {
    "pokemon.csv": "02eabc49d3c70bc1e150891bdb9acb254910adfdfc12a6020c7b7117b2eca32f",
    "baseball_stats.csv": "3741fc65a8bf1acbf022d872319e459127d89e6f42efdf0a586338729f0e724f",
}

def load_dataset(filename):
    context = ssl.create_default_context(cafile=certifi.where())
    with urlopen(DATASET_BASE + filename, context=context, timeout=60) as response:
        payload = response.read()
    assert hashlib.sha256(payload).hexdigest() == DATASET_HASHES[filename]
    frame = pd.read_csv(io.BytesIO(payload))
    print(filename, "shape:", frame.shape)
    return frame

print("Dataset loader ready; HTTPS certificates and SHA-256 hashes are verified.")


## Example 1


In [ ]:
# combining objects 
names = ['Ahmed', 'Youssef', 'Mohammed']
age = [25, 27, 40]
combined = []

for i,name in enumerate(names):
    combined.append((name, age[i]))
print(combined)


## Example 2


In [ ]:
# Combining objects with zip
combined_zip = zip(names, age)
print(type(combined_zip))


## Example 3


In [ ]:
combined_zip_list = [*combined_zip]
print(combined_zip_list)


## Example 4


In [ ]:
pokemon = load_dataset("pokemon.csv")
print(pokemon.head().to_string(index=False))


## Example 5


In [ ]:
poke_types = pokemon['Type 1']

def count_with_loop(values):
    type_counts = {}
    for poke_type in values:
        if poke_type not in type_counts:
            type_counts[poke_type] = 1
        else:
            type_counts[poke_type] += 1
    return type_counts

loop_counts = count_with_loop(poke_types)
print(loop_counts)
%timeit -r3 -n100 count_with_loop(poke_types)


## Example 6


In [ ]:
from collections import Counter

counter_counts = Counter(poke_types)
print(counter_counts)
%timeit -r3 -n100 Counter(poke_types)


## Example 7


In [ ]:
#Combinations with loop

poke_types = ['Bug', 'Fire', 'Ghost', 'Grass', 'Water']
combos = []
for x in poke_types:
    for y in poke_types:
        if x == y:
            continue
        if ((x,y) not in combos) & ((y,x) not in combos):
            combos.append((x,y))
print(combos)


## Example 8


In [ ]:
# combinations with itertools
poke_types = ['Bug', 'Fire', 'Ghost', 'Grass', 'Water']
from itertools import combinations
combos_obj = combinations(poke_types, 2)
print(type(combos_obj))

combos = [*combos_obj]
print(combos)


## Example 9


In [ ]:
# Comparing objects with loops
list_a = ['Bulbasaur', 'Charmander', 'Squirtle']
list_b = ['Caterpie', 'Pidgey', 'Squirtle']

in_common = []
for pokemon_a in list_a:
    for pokemon_b in list_b:
        if pokemon_a == pokemon_b:
            in_common.append(pokemon_a)
print(in_common)


## Example 10


In [ ]:
# compaing objects using intersection()

list_a = ['Bulbasaur', 'Charmander', 'Squirtle']
list_b = ['Caterpie', 'Pidgey', 'Squirtle']
set_a = set(list_a)
set_b = set(list_b)
print(set_a.intersection(set_b))


## Example 11


In [ ]:
def intersection_with_loop(left, right):
    in_common = []
    for pokemon_a in left:
        for pokemon_b in right:
            if pokemon_a == pokemon_b:
                in_common.append(pokemon_a)
    return in_common

print(intersection_with_loop(list_a, list_b))
%timeit -r3 -n1000 intersection_with_loop(list_a, list_b)


## Example 12


In [ ]:
print(set_a.intersection(set_b))
%timeit -r3 -n1000 set_a.intersection(set_b)


## Example 13


In [ ]:
set_a = {'Bulbasaur', 'Charmander', 'Squirtle'}
set_b = {'Caterpie', 'Pidgey', 'Squirtle'}
print(set_a.difference(set_b))


## Example 14


In [ ]:
set_a = {'Bulbasaur', 'Charmander', 'Squirtle'}
set_b = {'Caterpie', 'Pidgey', 'Squirtle'}
print(set_b.difference(set_a))


## Example 15


In [ ]:
set_a = {'Bulbasaur', 'Charmander', 'Squirtle'}
set_b = {'Caterpie', 'Pidgey', 'Squirtle'}
print(set_a.symmetric_difference(set_b))


## Example 16


In [ ]:
set_a = {'Bulbasaur', 'Charmander', 'Squirtle'}
set_b = {'Caterpie', 'Pidgey', 'Squirtle'}
print(set_a.union(set_b))


## Example 17


In [ ]:
names_list = list(pokemon['Name'])
names_set = set(pokemon['Name'])
names_tuple = tuple(pokemon['Name'])
print(len(names_list), len(names_tuple), len(names_set))


## Example 18


In [ ]:
%timeit -r3 -n1000 'Zubat' in names_list
%timeit -r3 -n1000 'Zubat' in names_tuple
%timeit -r3 -n1000 'Zubat' in names_set


## Example 19


In [ ]:
primary_types = list(pokemon['Type 1'])
unique_types = []
for prim_type in primary_types:
    if prim_type not in unique_types:
        unique_types.append(prim_type)
print(unique_types)


## Example 20


In [ ]:
unique_types_set = set(primary_types)
print(sorted(unique_types_set))


## Example 21


In [ ]:
# List of HP, Attack, Defense, Speed
poke_stats = [
[90, 92, 75, 60],
[25, 20, 15, 90],
[65, 130, 60, 75],
]
print(poke_stats)


## Example 22


In [ ]:
def sum_rows_with_loop(rows):
    totals = []
    for row in rows:
        totals.append(sum(row))
    return totals

print(sum_rows_with_loop(poke_stats))
%timeit -r3 -n1000 sum_rows_with_loop(poke_stats)


## Example 23


In [ ]:
totals_comp = [sum(row) for row in poke_stats]
print(totals_comp)
%timeit -r3 -n1000 [sum(row) for row in poke_stats]


## Example 24


In [ ]:
totals_map = [*map(sum, poke_stats)]
print(totals_map)
%timeit -r3 -n1000 [*map(sum, poke_stats)]


## Example 25


In [ ]:
poke_types = ['Bug', 'Fire', 'Ghost', 'Grass', 'Water']

# Nested for loop approach
combos = []
for x in poke_types:
    for y in poke_types:
        if x == y:
            continue
        if ((x,y) not in combos) & ((y,x) not in combos):
            combos.append((x,y))
print(combos)


## Example 26


In [ ]:
# Built-in module approach
from itertools import combinations
combos2 = [*combinations(poke_types, 2)]
print(combos2)


## Example 27


In [ ]:
poke_stats = np.asarray(poke_stats)

def mean_rows_with_loop(rows):
    avgs = []
    for row in rows:
        avg = np.mean(row)
        avgs.append(avg)
    return avgs

print(mean_rows_with_loop(poke_stats))
%timeit -r3 -n1000 mean_rows_with_loop(poke_stats)


## Example 28


In [ ]:
avgs = poke_stats.mean(axis=1)
print(avgs)
%timeit -r3 -n1000 poke_stats.mean(axis=1)


## Example 29


In [ ]:
import numpy as np

names = ['Absol', 'Aron', 'Jynx', 'Natu', 'Onix']
attacks = np.array([130, 70, 50, 50, 45])

for pokemon_name,attack in zip(names, attacks):
    total_attack_avg = attacks.mean()
    if attack > total_attack_avg:
        print(
            "{}'s attack: {} > average: {}!"
            .format(pokemon_name, attack, total_attack_avg)
        )


## Example 30


In [ ]:
import numpy as np

names = ['Absol', 'Aron', 'Jynx', 'Natu', 'Onix']
attacks = np.array([130, 70, 50, 50, 45])

# Calculate total average once (outside the loop)
total_attack_avg = attacks.mean()

for pokemon_name,attack in zip(names, attacks):
    if attack > total_attack_avg:
        print(
            "{}'s attack: {} > average: {}!"
            .format(pokemon_name, attack, total_attack_avg)
        )


## Example 31


In [ ]:
def attacks_with_repeated_mean(names, attacks):
    messages = []
    for pokemon, attack in zip(names, attacks):
        total_attack_avg = attacks.mean()
        if attack > total_attack_avg:
            messages.append("{}'s attack: {} > average: {}!".format(pokemon, attack, total_attack_avg))
    return messages

print(*attacks_with_repeated_mean(names, attacks), sep="\n")
%timeit -r3 -n1000 attacks_with_repeated_mean(names, attacks)


## Example 32


In [ ]:
def attacks_with_one_mean(names, attacks):
    messages = []
    total_attack_avg = attacks.mean()
    for pokemon, attack in zip(names, attacks):
        if attack > total_attack_avg:
            messages.append("{}'s attack: {} > average: {}!".format(pokemon, attack, total_attack_avg))
    return messages

print(*attacks_with_one_mean(names, attacks), sep="\n")
%timeit -r3 -n1000 attacks_with_one_mean(names, attacks)


## Example 33


In [ ]:
legend_status_list = pokemon['Legendary'].tolist()
generations_list = pokemon['Generation'].tolist()

def convert_in_loop(names, legends, generations):
    poke_data = []
    for poke_tuple in zip(names, legends, generations):
        poke_list = list(poke_tuple)
        poke_data.append(poke_list)
    return poke_data

poke_data = convert_in_loop(names_list, legend_status_list, generations_list)
print(poke_data[:5])
%timeit -r3 -n100 convert_in_loop(names_list, legend_status_list, generations_list)


## Example 34


In [ ]:
def convert_after_loop(names, legends, generations):
    poke_data_tuples = []
    for poke_tuple in zip(names, legends, generations):
        poke_data_tuples.append(poke_tuple)
    return [*map(list, poke_data_tuples)]

poke_data_holistic = convert_after_loop(names_list, legend_status_list, generations_list)
print(poke_data_holistic[:5])
%timeit -r3 -n100 convert_after_loop(names_list, legend_status_list, generations_list)


## Example 35


In [ ]:
baseball_df = load_dataset("baseball_stats.csv")
print(baseball_df.head().to_string(index=False))


## Example 36


In [ ]:
def calc_win_perc(wins, games_played):
    win_perc = wins / games_played
    return np.round(win_perc,2)
print(calc_win_perc(81, 162))


## Example 37


In [ ]:
def win_percentages_iloc(frame):
    win_perc_list = []
    for i in range(len(frame)):
        row = frame.iloc[i]
        win_perc_list.append(calc_win_perc(row['W'], row['G']))
    return win_perc_list

baseball_df['WP'] = win_percentages_iloc(baseball_df)
print(baseball_df[['Team', 'Year', 'W', 'G', 'WP']].head().to_string(index=False))
%timeit -r3 -n1 win_percentages_iloc(baseball_df)


## Example 38


In [ ]:
def win_percentages_iterrows(frame):
    win_perc_list = []
    for i, row in frame.iterrows():
        win_perc_list.append(calc_win_perc(row['W'], row['G']))
    return win_perc_list

baseball_df['WP'] = win_percentages_iterrows(baseball_df)
print(baseball_df[['Team', 'Year', 'W', 'G', 'WP']].head().to_string(index=False))
%timeit -r3 -n1 win_percentages_iterrows(baseball_df)


## Example 39


In [ ]:
team_wins_df = baseball_df[['Team', 'Year', 'W']]
for row_namedtuple in team_wins_df.head().itertuples():
    print(row_namedtuple)


## Example 40


In [ ]:
print(row_namedtuple.Index)


## Example 41


In [ ]:
print(row_namedtuple.Team)


## Example 42


In [ ]:
def rows_as_series(frame):
    return [row_tuple for row_tuple in frame.iterrows()]

print(rows_as_series(team_wins_df.head(1))[0])
%timeit -r3 -n1 rows_as_series(team_wins_df)


## Example 43


In [ ]:
def rows_as_namedtuples(frame):
    return [row_namedtuple for row_namedtuple in frame.itertuples()]

print(rows_as_namedtuples(team_wins_df.head(1))[0])
%timeit -r3 -n1 rows_as_namedtuples(team_wins_df)


## Example 44


In [ ]:
def calc_run_diff(runs_scored, runs_allowed):
    run_diff = runs_scored - runs_allowed
    return run_diff
print(calc_run_diff(734, 688))


## Run differences with iterrows


In [ ]:
def run_differences_iterrows(frame):
    run_diffs_iterrows = []
    for i, row in frame.iterrows():
        run_diffs_iterrows.append(calc_run_diff(row['RS'], row['RA']))
    return run_diffs_iterrows

run_diffs_iterrows = run_differences_iterrows(baseball_df)
baseball_df['RD'] = run_diffs_iterrows
print(baseball_df[['Team', 'Year', 'RS', 'RA', 'RD']].head().to_string(index=False))
%timeit -r3 -n1 run_differences_iterrows(baseball_df)


## Example 45


In [ ]:
def run_differences_apply(frame):
    return frame.apply(lambda row: calc_run_diff(row['RS'], row['RA']), axis=1)

run_diffs_apply = run_differences_apply(baseball_df)
baseball_df['RD'] = run_diffs_apply
print(baseball_df[['Team', 'Year', 'RS', 'RA', 'RD']].head().to_string(index=False))
%timeit -r3 -n1 run_differences_apply(baseball_df)


## Example 46


In [ ]:
wins_np = baseball_df['W'].values
print(type(wins_np))


## Example 47


In [ ]:
print(wins_np)


## Example 48


In [ ]:
def run_differences_numpy(frame):
    return frame['RS'].values - frame['RA'].values

run_diffs_np = run_differences_numpy(baseball_df)
baseball_df['RD'] = run_diffs_np
print(baseball_df[['Team', 'Year', 'RS', 'RA', 'RD']].head().to_string(index=False))
%timeit -r3 -n100 run_differences_numpy(baseball_df)


## Regression checks

Check full original inputs, equivalent results, reordered indices and unchanged frames. No absolute timings or speed rankings are asserted.


In [ ]:
from itertools import combinations
from pandas.testing import assert_frame_equal

assert pokemon.shape == (800, 13)
assert baseball_df.shape == (1232, 17)
assert list(combined_zip_list) == [('Ahmed', 25), ('Youssef', 27), ('Mohammed', 40)]
assert loop_counts == dict(counter_counts) == pokemon['Type 1'].value_counts().to_dict()
assert sum(loop_counts.values()) == 800
assert len(loop_counts) == 18
assert combos == combos2 == list(combinations(['Bug', 'Fire', 'Ghost', 'Grass', 'Water'], 2))
assert len(combos) == 10
assert intersection_with_loop(list_a, list_b) == ['Squirtle']
assert set_a & set_b == {'Squirtle'}
assert set_a - set_b == {'Bulbasaur', 'Charmander'}
assert set_b - set_a == {'Caterpie', 'Pidgey'}
assert set_a ^ set_b == {'Bulbasaur', 'Charmander', 'Caterpie', 'Pidgey'}
assert set_a | set_b == {'Bulbasaur', 'Charmander', 'Squirtle', 'Caterpie', 'Pidgey'}
assert all('Zubat' in values for values in [names_list, names_tuple, names_set])
assert set(unique_types) == unique_types_set == set(pokemon['Type 1'])
assert sum_rows_with_loop(poke_stats) == totals_comp == totals_map == [317, 150, 330]
np.testing.assert_allclose(mean_rows_with_loop(poke_stats), avgs)
np.testing.assert_allclose(avgs, [79.25, 37.5, 82.5])
assert attacks_with_repeated_mean(names, attacks) == attacks_with_one_mean(names, attacks)
assert len(attacks_with_one_mean(names, attacks)) == 2
assert poke_data == poke_data_holistic
assert len(poke_data) == 800
assert poke_data == pokemon[['Name', 'Legendary', 'Generation']].values.tolist()

before = baseball_df.copy(deep=True)
expected_wp = np.round(baseball_df['W'].to_numpy() / baseball_df['G'].to_numpy(), 2)
np.testing.assert_array_equal(win_percentages_iloc(baseball_df), expected_wp)
np.testing.assert_array_equal(win_percentages_iterrows(baseball_df), expected_wp)
np.testing.assert_array_equal(baseball_df['WP'], expected_wp)
assert expected_wp[:5].tolist() == [0.5, 0.58, 0.57, 0.43, 0.38]
expected_rd = baseball_df['RS'].to_numpy() - baseball_df['RA'].to_numpy()
np.testing.assert_array_equal(run_differences_iterrows(baseball_df), expected_rd)
np.testing.assert_array_equal(run_differences_apply(baseball_df), expected_rd)
np.testing.assert_array_equal(run_differences_numpy(baseball_df), expected_rd)
np.testing.assert_array_equal(baseball_df['RD'], expected_rd)
assert expected_rd[:5].tolist() == [46, 100, 7, -72, -146]
assert [tuple(series) for _, series in rows_as_series(team_wins_df)] == [tuple(row)[1:] for row in rows_as_namedtuples(team_wins_df)]
assert_frame_equal(baseball_df, before)

# Equivalence must not depend on a RangeIndex or original row order.
shuffled = baseball_df.sample(frac=1, random_state=7)
np.testing.assert_array_equal(win_percentages_iloc(shuffled), win_percentages_iterrows(shuffled))
np.testing.assert_array_equal(run_differences_iterrows(shuffled), run_differences_numpy(shuffled))
np.testing.assert_array_equal(run_differences_apply(shuffled), run_differences_numpy(shuffled))
empty = baseball_df.iloc[:0]
assert win_percentages_iloc(empty) == win_percentages_iterrows(empty) == []
assert run_differences_iterrows(empty) == []
assert run_differences_numpy(empty).size == 0
assert count_with_loop([]) == {}
assert convert_in_loop([], [], []) == convert_after_loop([], [], []) == []
print("All full-dataset, set, combination, loop, vectorization, index and input-preservation checks passed.")
print("Pokémon records:", len(pokemon), "types:", len(counter_counts))
print("Baseball records:", len(baseball_df), "years:", (int(baseball_df.Year.min()), int(baseball_df.Year.max())))
print("First five win percentages:", expected_wp[:5].tolist())
print("First five run differentials:", expected_rd[:5].tolist())
